# MCP Primitives: The Three Things a Server Can Offer

Every MCP server, no matter what it connects to, can offer exactly three kinds of
things to the client talking to it. This notebook explains each one with a real,
familiar tool as the example, then builds all three into one small, real server --
proving, in actual working code, just how little it takes to go from a tools-only
server to one that offers everything.


## What's a Primitive?

Picture a gadget connected to a universal remote. That gadget can do exactly three
kinds of things: something it *does* when a button is pressed, something you can just
*glance at* like a small status display, and a *suggested routine* the manufacturer
built in ahead of time -- a "movie night" preset, say. Those three things map directly
onto MCP's three primitives:

- **Tools** -- actions the AI asks the server to perform
- **Resources** -- structured data the AI can read
- **Prompts** -- ready-made templates the server offers to shape how the AI works

```mermaid
flowchart LR
 T[Tools: something it DOES] ~~~ R[Resources: something to READ] ~~~ P[Prompts: a TEMPLATE to follow]
```


## Tools: Grounded in GitHub

A **tool** is an action with a real effect -- not just information coming back, but
something actually happening as a result of the call. Picture a GitHub-connected MCP
server. It exposes a tool called `create_issue`. Nobody calls this tool by hand -- the
AI decides to, the moment it judges from the conversation that logging a bug is the
right next step.

Two protocol operations cover everything a client needs: `tools/list` to discover what
a server offers, and `tools/call` to actually run one.

```mermaid
sequenceDiagram
 participant AI
 participant GH as GitHub Server
 AI->>GH: tools/list
 GH-->>AI: ["create_issue", ...]
 AI->>GH: tools/call(create_issue, {title, body})
 GH-->>AI: issue created, #482
```

A tool is the one primitive where the AI itself is holding the wheel. Every other
primitive is initiated by someone else first.


## Resources: Grounded in Google Drive

A **resource** is read-only reference data -- something to look at, never something
that changes anything. Picture a Google-Drive-connected MCP server exposing a resource
for one specific file: the team's style guide. The *application* -- not the model --
decides to read that resource and hand its content in as context, so answers stay
consistent with house style without anyone re-typing the guide into every conversation.

`resources/list` discovers what's available; `resources/read` actually fetches it.

```mermaid
sequenceDiagram
 participant App as Application
 participant Drv as Drive Server
 App->>Drv: resources/list
 Drv-->>App: ["style-guide.docx", ...]
 App->>Drv: resources/read(style-guide.docx)
 Drv-->>App: full file content
```

Resources can do more than a single file -- **templates** let one resource answer for
a whole family of things (one template covering every customer record, say, instead of
listing each one), and **subscriptions** let a client get notified the moment a specific
resource changes instead of re-reading it on a timer. This notebook keeps its own
example to one plain file; the official MCP documentation covers templates and
subscriptions in full when that need actually arises.


## Prompts: A Template, Shown Before and After

A **prompt** is a ready-made template the server offers, so a request gets structured
the same way every time instead of being improvised from scratch. `prompts/list`
discovers what's offered; `prompts/get` pulls a prompt's full details.

Here's the exact template this course has been building since Part 2 -- a customer
escalation prompt -- shown as its definition and then as what it actually produces.


In [ ]:
prompt_definition = {
    "name": "structured_escalation",
    "description": "Guides the AI to log a customer escalation with every required field",
    "arguments": [
        {"name": "issue_summary", "required": True},
        {"name": "what_was_tried", "required": True},
        {"name": "customer_sentiment", "required": True},
    ]
}

# What it looks like filled in, every single time it's used:
example_output = """
Issue Summary: Order #4521 arrived damaged
What Was Already Tried: Customer emailed support once, no reply in 3 days
Customer Sentiment: Frustrated, considering a refund request
Recommended Next Action: Escalate to logistics team, offer expedited replacement
"""

import json
print('THE TEMPLATE:')
print(json.dumps(prompt_definition, indent=2))
print()
print('WHAT IT PRODUCES:')
print(example_output)


Nothing about the underlying model changed between a vague, unstructured note and this
output. The server simply handed it a form to fill in -- that's the entire value of a
prompt primitive.


## All Three, in Real Code — Step by Step

This is the part most tutorials skip: exactly how little code separates a tools-only
server from one offering all three primitives. Starting from the warm-up server built
back in Part 2 -- which already has `greet` and `add` as tools -- here is every step
needed to add a real resource and a real prompt.


In [ ]:
# Step 0: the server as it already exists, from Part 2 -- tools only
step0 = '''
from fastmcp import FastMCP

mcp = FastMCP("Warm-Up Server")

@mcp.tool
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}! Welcome to MCP."

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b
'''
print(step0)


In [ ]:
# Step 1: add ONE resource -- a real, local file being read
step1_addition = '''
@mcp.resource("file://server-notes")
def server_notes() -> str:
    """Read-only notes about this server, straight from a local file."""
    with open("server-notes.txt") as f:
        return f.read()
'''
print(step1_addition)


In [ ]:
# Step 2: add ONE prompt -- the escalation template from above, now as real code
step2_addition = '''
@mcp.prompt
def structured_escalation(issue_summary: str, what_was_tried: str, customer_sentiment: str) -> str:
    """Guides the AI to log a customer escalation with every required field, in order."""
    return f"""Log this customer escalation with the following structure:
Issue Summary: {issue_summary}
What Was Already Tried: {what_was_tried}
Customer Sentiment: {customer_sentiment}
Recommended Next Action: [determine this from the details above]
"""
'''
print(step2_addition)


In [ ]:
# Assemble the complete, final server and write it to disk
final_server_code = '''
from fastmcp import FastMCP

mcp = FastMCP("Warm-Up Server")

@mcp.tool
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}! Welcome to MCP."

@mcp.resource("file://server-notes")
def server_notes() -> str:
    """Read-only notes about this server, straight from a local file."""
    with open("server-notes.txt") as f:
        return f.read()

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@mcp.prompt
def structured_escalation(issue_summary: str, what_was_tried: str, customer_sentiment: str) -> str:
    """Guides the AI to log a customer escalation with every required field, in order."""
    return f"""Log this customer escalation with the following structure:
Issue Summary: {issue_summary}
What Was Already Tried: {what_was_tried}
Customer Sentiment: {customer_sentiment}
Recommended Next Action: [determine this from the details above]
"""

if __name__ == "__main__":
    mcp.run()
'''

with open('server.py', 'w') as f:
    f.write(final_server_code)

with open('server-notes.txt', 'w') as f:
    f.write('This server was built across Parts 2 and 4A of the MCP crash course.\n'
            'It demonstrates all three primitives: tools, resources, and prompts.')

print(final_server_code)


Run `uv run fastmcp dev server.py` and open Inspector: `greet` and `add` are still
there under Tools, `file://server-notes` now appears under Resources and returns the
real file's content, and `structured_escalation` now appears under Prompts, asking for
the same three fields shown earlier in this notebook.

**Two decorators. That's the entire distance between a tools-only server and one that
offers all three primitives.** Most tutorials stop at tools because tools are the
easiest thing to demo -- the honest truth is resources and prompts are just as easy to
build.


## Summary

| Primitive | One-line definition | Protocol operations | Real example |
|---|---|---|---|
| Tools | Actions the AI asks the server to perform | `tools/list`, `tools/call` | GitHub's `create_issue` |
| Resources | Structured data the AI can read | `resources/list`, `resources/read` | Google Drive's style guide |
| Prompts | Ready-made templates that shape the request | `prompts/list`, `prompts/get` | `structured_escalation` |

```mermaid
flowchart TB
 S[server.py] -->|@mcp.tool| T[greet, add]
 S -->|@mcp.resource| R[file://server-notes]
 S -->|@mcp.prompt| P[structured_escalation]
```

**Next:** all of this, used for real, in one complete live conversation -- from the very
first message to the very last.
